In [ ]:
import pandas as pd

# قراءة ملف CSV
df = pd.read_csv("panda70m_testing_with_additional_annotation.csv")



# اختيار 1000 عينة
df_1000 = df.head(1000)

# حفظ ملف جديد
df_1000.to_csv("panda_test_1000.csv", index=False)

print("عدد العينات:", len(df_1000))


In [ ]:
!pip install yt-dlp


In [ ]:
import pandas as pd
import os
import subprocess

# قراءة الملف
df = pd.read_csv("panda_test_1000.csv")

# أخذ 50 عينات فقط للاختبار
df_test = df.head(50)

# إنشاء مجلد الفيديوهات
os.makedirs("data/videos", exist_ok=True)

for i, row in df_test.iterrows():
    video_id = row["videoID"]
    url = row["url"]
    output_path = f"data/videos/{video_id}.mp4"

    if os.path.exists(output_path):
        print(f"Already exists: {video_id}")
        continue

    command = [
        "yt-dlp",
        "--max-filesize", "100M",
        "-f", "mp4",
        "-o", output_path,
        url
    ]

    try:
        subprocess.run(command, check=True)
        print(f"Downloaded: {video_id}")
    except Exception:
        print(f"Skipped (too large or failed): {video_id}")



In [ ]:
df_test = df.head(1000)

In [ ]:
import cv2
import os
import numpy as np

# تعريف المسارات (مهم جدًا)
video_dir = "data/videos"
frames_dir = "frames"

os.makedirs(frames_dir, exist_ok=True)

NUM_FRAMES = 12

for video_file in os.listdir(video_dir):
    if not video_file.endswith(".mp4"):
        continue

    video_id = video_file.replace(".mp4", "")
    video_path = os.path.join(video_dir, video_file)

    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        cap.release()
        continue

    frame_indices = np.linspace(0, total_frames - 1, NUM_FRAMES).astype(int)

    video_frames_dir = os.path.join(frames_dir, video_id)
    os.makedirs(video_frames_dir, exist_ok=True)

    idx_set = set(frame_indices)
    frame_id = 0
    saved = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_id in idx_set:
            cv2.imwrite(
                os.path.join(video_frames_dir, f"frame_{saved:04d}.jpg"),
                frame
            )
            saved += 1

        frame_id += 1

    cap.release()
    print(f"{video_id}: saved {saved} frames")


In [ ]:
!pip install torch torchvision ftfy regex tqdm
!pip install git+https://github.com/openai/CLIP.git

In [ ]:
import torch
import clip
import os
import cv2
import numpy as np
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"

# تحميل نموذج CLIP
model, preprocess = clip.load("ViT-B/32", device=device)

frames_root = "frames"
video_embeddings = {}

for video_id in os.listdir(frames_root):
    video_path = os.path.join(frames_root, video_id)
    if not os.path.isdir(video_path):
        continue

    frame_embeddings = []

    for frame_file in sorted(os.listdir(video_path)):

        frame_path = os.path.join(video_path, frame_file)

        image = Image.open(frame_path).convert("RGB")
        image_input = preprocess(image).unsqueeze(0).to(device)

        with torch.no_grad():
            embedding = model.encode_image(image_input)
            embedding = embedding / embedding.norm(dim=-1, keepdim=True)

        frame_embeddings.append(embedding.cpu().numpy())

    if len(frame_embeddings) > 0:
    # تحويل القائمة إلى مصفوفة (num_frames, 1, 512)
        frame_embeddings_np = np.vstack(frame_embeddings)

    # Max Pooling عبر الإطارات
        video_embedding = np.max(frame_embeddings_np, axis=0)

    # إزالة البعد الزائد (1,512) → (512,)
        video_embedding = video_embedding.squeeze()

    # Normalize
        video_embedding = video_embedding / np.linalg.norm(video_embedding)

        video_embeddings[video_id] = video_embedding

    print(f"{video_id}: {len(frame_embeddings)} frames encoded")


In [ ]:
import pandas as pd
import torch
import clip

device = "cuda" if torch.cuda.is_available() else "cpu"



text_embeddings = {}

for _, row in df_test.iterrows():
    video_id = row["videoID"]
    caption = row["caption"]

    text_input = clip.tokenize([caption], truncate=True).to(device)

    with torch.no_grad():
        embedding = model.encode_text(text_input)
        embedding = embedding / embedding.norm(dim=-1, keepdim=True)

    text_embeddings[video_id] = embedding.cpu().numpy()


In [ ]:
import numpy as np

alpha = 0.9  # وزن التمثيل البصري (يمكن تغييره)
fused_video_embeddings = {}

for video_id, visual_emb in video_embeddings.items():

    # تأكد أن الفيديو لديه caption
    if video_id not in text_embeddings:
        continue

    text_emb = text_embeddings[video_id]

    # إزالة البعد الزائد (1,512) → (512,)
    visual_emb = visual_emb.squeeze()
    text_emb = text_emb.squeeze()

    # Late Fusion
    fused_emb = alpha * visual_emb + (1 - alpha) * text_emb

    # Normalize
    fused_emb = fused_emb / np.linalg.norm(fused_emb)

    fused_video_embeddings[video_id] = fused_emb

print("عدد الفيديوهات بعد الدمج:", len(fused_video_embeddings))


In [ ]:
import faiss

video_ids = list(fused_video_embeddings.keys())
embeddings_matrix = np.vstack(
    [fused_video_embeddings[vid] for vid in video_ids]
).astype("float32")

# تطبيع
faiss.normalize_L2(embeddings_matrix)

# إنشاء الفهرس
index = faiss.IndexFlatIP(embeddings_matrix.shape[1])
index.add(embeddings_matrix)

print("FAISS index built with fused embeddings")
def retrieve_videos(query, top_k=5):
    query_input = clip.tokenize([query], truncate=True).to(device)

    with torch.no_grad():
        query_emb = model.encode_text(query_input)
        query_emb = query_emb / query_emb.norm(dim=-1, keepdim=True)

    query_emb = query_emb.cpu().numpy().astype("float32")
    faiss.normalize_L2(query_emb)

    distances, indices = index.search(query_emb, top_k)

    results = [(video_ids[i], distances[0][j]) 
               for j, i in enumerate(indices[0])]
    return results


In [ ]:
query = "woman talking"
results = retrieve_videos(query, top_k=5)

print("Query:", query)
for vid, score in results:
    print(vid, round(score, 3))


In [ ]:
from IPython.display import HTML, display
import base64
top_video_id = results[0][0]

video_path = f"data/videos/{top_video_id}.mp4"

with open(video_path, "rb") as f:
    video_bytes = f.read()

video_base64 = base64.b64encode(video_bytes).decode("utf-8")

display(HTML(f"""
<video width="600" controls>
    <source src="data:video/mp4;base64,{video_base64}" type="video/mp4">
</video>
"""))

In [ ]:
# نقيّم فقط الفيديوهات التي تم تنزيلها فعليًا
valid_video_ids = set(video_embeddings.keys())

df_eval = df[df["videoID"].isin(valid_video_ids)].reset_index(drop=True)

print("عدد عينات التقييم:", len(df_eval))


In [ ]:
# دوال التقييم

def recall_at_k(retrieved_ids, relevant_id, k):
    return int(relevant_id in retrieved_ids[:k])

def precision_at_k(retrieved_ids, relevant_id, k):
    return int(relevant_id in retrieved_ids[:k]) / k




In [ ]:
# تنفيذ التقييم

Ks = [1, 5, 10]

recall_scores = {k: [] for k in Ks}
precision_scores = {k: [] for k in Ks}
mrr_scores = []

for _, row in df_eval.iterrows():
    query = row["caption"]
    gt_video = row["videoID"]

    results = retrieve_videos(query, top_k=max(Ks))
    retrieved_ids = [vid for vid, _ in results]

    for k in Ks:
        recall_scores[k].append(recall_at_k(retrieved_ids, gt_video, k))
        precision_scores[k].append(precision_at_k(retrieved_ids, gt_video, k))

    


In [ ]:
# عرض النتائج النهائية

print("=== Evaluation Results ===")

for k in Ks:
    print(f"Recall@{k}: {np.mean(recall_scores[k]):.3f}")
    print(f"Precision@{k}: {np.mean(precision_scores[k]):.3f}")



In [ ]:
# ============================================
# تقسيم البيانات إلى تدريب واختبار
# ============================================

from sklearn.model_selection import train_test_split
import pandas as pd

# قراءة البيانات
df = pd.read_csv("panda_test_1000.csv")

# 1. تصفية الفيديوهات التي تم تنزيلها فعلياً
valid_video_ids = set(video_embeddings.keys())
df_filtered = df[df["videoID"].isin(valid_video_ids)].reset_index(drop=True)

print(f"العدد الإجمالي للعينات المتاحة: {len(df_filtered)}")

# 2. تقسيم البيانات بنسبة 80% تدريب، 20% اختبار
train_df, test_df = train_test_split(
    df_filtered, 
    test_size=0.2, 
    random_state=42,  # للتكرارية
    stratify=None  # يمكن استخدامه إذا كانت هناك فئات
)

print(f"عدد عينات التدريب: {len(train_df)}")
print(f"عدد عينات الاختبار: {len(test_df)}")

# 3. حفظ المجموعات في ملفات منفصلة
train_df.to_csv("train_set.csv", index=False)
test_df.to_csv("test_set.csv", index=False)

In [ ]:
# التقييم النهائي يكون على test_df فقط
def evaluate_on_test_set(test_df, video_embeddings):
    """تقييم النظام على مجموعة الاختبار"""
    Ks = [1, 5, 10]
    recall_scores = {k: [] for k in Ks}
    
    for _, row in test_df.iterrows():
        query = row["caption"]
        gt_video = row["videoID"]
        
        # استرجاع الفيديوهات
        results = retrieve_videos(query, top_k=max(Ks))
        retrieved_ids = [vid for vid, _ in results]
        
        # حساب المقاييس
        for k in Ks:
            recall_scores[k].append(int(gt_video in retrieved_ids[:k]))
    
    # عرض النتائج
    print("=== نتائج التقييم على مجموعة الاختبار ===")
    for k in Ks:
        print(f"Recall@{k}: {np.mean(recall_scores[k]):.3f}")
    
    return recall_scores

# تنفيذ التقييم
test_results = evaluate_on_test_set(test_df, video_embeddings)